# 单图数据集通用探索

这个 notebook 直接调用各数据集目录中的 `gen_data.py`，用于统一检查 Cora、Pubmed、WikiCS 和其他单图数据集的图结构、文本特征与任务映射。

在下方配置单元修改 `DATASET_NAME`，或在启动 Jupyter 前设置环境变量 `SINGLE_GRAPH_DATASET`。需要下载原始图数据的数据集（例如 WikiCS、arxiv）会使用项目下的 `cache_data/<数据集>/raw` 目录。

In [5]:
from pathlib import Path
from types import SimpleNamespace
import importlib
import inspect
import os
import sys

import pandas as pd
import torch
from IPython.display import display


DATASET_NAME = os.environ.get("SINGLE_GRAPH_DATASET", "Cora")
SAMPLE_SIZE = 5
SPLIT_INDEX = 0  # 多组数据划分时选择第几组

## 1. 定位项目并选择数据集

In [6]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        marker = candidate / "data" / "single_graph" / "gen_data.py"
        if marker.is_file():
            return candidate
    raise FileNotFoundError(
        "无法定位项目根目录；请从仓库内启动 Jupyter。"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SINGLE_GRAPH_ROOT = PROJECT_ROOT / "data" / "single_graph"

available_datasets = sorted(
    path.name
    for path in SINGLE_GRAPH_ROOT.iterdir()
    if path.is_dir() and (path / "gen_data.py").is_file()
)
dataset_lookup = {name.lower(): name for name in available_datasets}

requested_dataset = DATASET_NAME.strip().lower()
if requested_dataset not in dataset_lookup:
    raise ValueError(
        f"未知数据集 {DATASET_NAME!r}；可选值为 {available_datasets}"
    )

DATASET_NAME = dataset_lookup[requested_dataset]
DATASET_DIR = SINGLE_GRAPH_ROOT / DATASET_NAME
CACHE_DIR = PROJECT_ROOT / "cache_data" / DATASET_NAME / "raw"

print(f"项目目录: {PROJECT_ROOT}")
print(f"可用数据集: {available_datasets}")
print(f"当前数据集: {DATASET_NAME}")
print(f"数据加载器: {DATASET_DIR / 'gen_data.py'}")

项目目录: D:\Bishe\GNN-Task_Relation
可用数据集: ['Cora', 'Pubmed', 'arxiv', 'wikics']
当前数据集: Cora
数据加载器: D:\Bishe\GNN-Task_Relation\data\single_graph\Cora\gen_data.py


## 2. 使用数据集原生加载器

In [7]:
project_root_text = str(PROJECT_ROOT)
if project_root_text not in sys.path:
    sys.path.insert(0, project_root_text)

module_name = f"data.single_graph.{DATASET_NAME}.gen_data"
dataset_module = importlib.import_module(module_name)
dataset_context = SimpleNamespace(
    name=DATASET_NAME,
    data_dir=str(CACHE_DIR),
)

graphs, text_features, task_map = dataset_module.get_data(dataset_context)
if not graphs:
    raise ValueError(f"{DATASET_NAME} 的加载器没有返回图对象")

graph = graphs[0]
print(f"已加载 {len(graphs)} 个图")
print(graph)

torch.Size([2, 5278])
已加载 1 个图
Data(x=[2708, 384], edge_index=[2, 5278], y=[2708], raw_text=[2708], label_names=[7], train_masks=[10], val_masks=[10], test_masks=[10], raw_texts=[2708], category_names=[2708])


## 3. 图对象概览

In [8]:
def value_shape(value):
    shape = getattr(value, "shape", None)
    return tuple(shape) if shape is not None else None


def value_length(value):
    try:
        return len(value)
    except TypeError:
        return None


graph_values = graph.to_dict()
field_rows = []
for name, value in graph_values.items():
    field_rows.append(
        {
            "字段": name,
            "类型": type(value).__name__,
            "shape": value_shape(value),
            "dtype": str(getattr(value, "dtype", "")),
            "长度": value_length(value),
        }
    )

labels = getattr(graph, "y", None)
flat_labels = (
    labels.detach().cpu().reshape(-1)
    if isinstance(labels, torch.Tensor)
    else None
)
num_classes = (
    int(torch.unique(flat_labels).numel())
    if flat_labels is not None and flat_labels.numel()
    else None
)

summary = pd.Series(
    {
        "dataset": DATASET_NAME,
        "graph_type": type(graph).__name__,
        "num_nodes": int(graph.num_nodes),
        "num_edges": int(graph.num_edges),
        "num_classes": num_classes,
        "num_graph_fields": len(graph_values),
    },
    name="值",
)
display(summary.to_frame())
display(pd.DataFrame(field_rows))

,值
dataset,Cora
graph_type,Data
num_nodes,2708
num_edges,5278
num_classes,7
num_graph_fields,10


,字段,类型,shape,dtype,长度
0,x,Tensor,"(2708, 384)",torch.float32,2708
1,edge_index,Tensor,"(2, 5278)",torch.int64,2
2,y,Tensor,"(2708,)",torch.int64,2708
3,raw_text,list,None,,2708
4,label_names,list,None,,7
5,train_masks,list,None,,10
6,val_masks,list,None,,10
7,test_masks,list,None,,10
8,raw_texts,list,None,,2708
9,category_names,list,None,,2708


## 4. 文本特征

In [20]:
TEXT_FEATURE_NAMES = (
    "node_text_feat",
    "edge_text_feat",
    "noi_node_text_feat",
    "class_node_text_feat",
    "prompt_edge_text_feat",
)

if len(text_features) != len(TEXT_FEATURE_NAMES):
    raise ValueError(
        f"预期 {len(TEXT_FEATURE_NAMES)} 组文本特征，实际得到 {len(text_features)} 组"
    )

text_feature_map = dict(zip(TEXT_FEATURE_NAMES, text_features))
text_rows = []
for name, values in text_feature_map.items():
    sample = str(values[0])[:240] if len(values) else ""
    text_rows.append({"特征组": name, "数量": len(values), "首条样例": sample})

display(pd.DataFrame(text_rows))

,特征组,数量,首条样例
0,node_text_feat,11701,feature node. wikipedia entry name: twilio. en...
1,edge_text_feat,1,feature edge. wikipedia page link
2,noi_node_text_feat,1,prompt node. node classification of wikipedia ...
3,class_node_text_feat,10,prompt node. wikipedia entry category: computa...
4,prompt_edge_text_feat,1,prompt edge.


In [21]:
node_texts = text_feature_map["node_text_feat"]
class_texts = text_feature_map["class_node_text_feat"]
preview_size = min(SAMPLE_SIZE, len(node_texts), int(graph.num_nodes))

node_rows = []
for node_id in range(preview_size):
    label_id = (
        int(flat_labels[node_id])
        if flat_labels is not None and node_id < flat_labels.numel()
        else None
    )
    class_text = (
        str(class_texts[label_id])[:180]
        if label_id is not None and 0 <= label_id < len(class_texts)
        else ""
    )
    node_rows.append(
        {
            "节点 ID": node_id,
            "标签 ID": label_id,
            "节点文本": str(node_texts[node_id])[:240],
            "节点文本长度": len(str(node_texts[node_id])),
            "类别文本": class_text,
        }
    )

display(pd.DataFrame(node_rows))

,节点 ID,标签 ID,节点文本,节点文本长度,类别文本
0,0,7,feature node. wikipedia entry name: twilio. en...,3525,prompt node. wikipedia entry category: distrib...
1,1,2,feature node. wikipedia entry name: program_co...,474,prompt node. wikipedia entry category: operati...
2,2,2,feature node. wikipedia entry name: systat_(de...,285,prompt node. wikipedia entry category: operati...
3,3,1,feature node. wikipedia entry name: list_of_co...,175,prompt node. wikipedia entry category: databases
4,4,4,feature node. wikipedia entry name: stealth_wa...,792,prompt node. wikipedia entry category: compute...


## 5. 标签分布

In [8]:
if flat_labels is None or not flat_labels.numel():
    print("图对象没有可统计的 y 标签。")
else:
    label_ids, label_counts = torch.unique(
        flat_labels, sorted=True, return_counts=True
    )
    label_rows = []
    for label_id, count in zip(label_ids.tolist(), label_counts.tolist()):
        class_text = (
            str(class_texts[label_id])[:240]
            if 0 <= label_id < len(class_texts)
            else ""
        )
        label_rows.append(
            {"标签 ID": label_id, "节点数": count, "类别文本": class_text}
        )
    display(pd.DataFrame(label_rows))

,标签 ID,节点数,类别文本
0,0,180,prompt node. literature category and descripti...
1,1,818,prompt node. literature category and descripti...
2,2,298,prompt node. literature category and descripti...
3,3,418,prompt node. literature category and descripti...
4,4,351,prompt node. literature category and descripti...
5,5,217,prompt node. literature category and descripti...
6,6,426,prompt node. literature category and descripti...


## 6. 训练、验证与测试划分

In [ ]:
SPLIT_FIELD_NAMES = {
    "train": ("train_masks", "train_mask"),
    "val": ("val_masks", "val_mask"),
    "test": ("test_masks", "test_mask"),
}
SPLIT_DICT_KEYS = {
    "train": ("train",),
    "val": ("val", "valid", "validation"),
    "test": ("test",),
}


def select_mask(value, split_index, node_count, source_name):
    selected_index = 0
    available_splits = 1

    if isinstance(value, (list, tuple)):
        available_splits = len(value)
        if available_splits == 0:
            raise ValueError(f"{source_name} 为空")
        selected_index = split_index if available_splits > 1 else 0
        if not 0 <= selected_index < available_splits:
            raise IndexError(
                f"SPLIT_INDEX={split_index} 超出 {source_name} 的范围 "
                f"[0, {available_splits - 1}]"
            )
        value = value[selected_index]

    tensor = (
        value.detach().cpu()
        if isinstance(value, torch.Tensor)
        else torch.as_tensor(value)
    )

    if tensor.dtype == torch.bool:
        if tensor.ndim == 1:
            if tensor.numel() != node_count:
                raise ValueError(
                    f"{source_name} 长度为 {tensor.numel()}，节点数为 {node_count}"
                )
            return tensor, selected_index, available_splits

        if tensor.ndim == 2:
            if tensor.shape[0] == node_count:
                available_splits = tensor.shape[1]
                selected_index = split_index if available_splits > 1 else 0
                if not 0 <= selected_index < available_splits:
                    raise IndexError(
                        f"SPLIT_INDEX={split_index} 超出 {source_name} 的范围 "
                        f"[0, {available_splits - 1}]"
                    )
                return tensor[:, selected_index], selected_index, available_splits

            if tensor.shape[1] == node_count:
                available_splits = tensor.shape[0]
                selected_index = split_index if available_splits > 1 else 0
                if not 0 <= selected_index < available_splits:
                    raise IndexError(
                        f"SPLIT_INDEX={split_index} 超出 {source_name} 的范围 "
                        f"[0, {available_splits - 1}]"
                    )
                return tensor[selected_index], selected_index, available_splits

        raise ValueError(
            f"无法把 {source_name} 的 shape={tuple(tensor.shape)} 解释为节点 mask"
        )

    indices = tensor.to(torch.long).reshape(-1)
    if indices.numel() and (int(indices.min()) < 0 or int(indices.max()) >= node_count):
        raise IndexError(f"{source_name} 中存在越界节点索引")
    mask = torch.zeros(node_count, dtype=torch.bool)
    mask[indices] = True
    return mask, selected_index, available_splits


def resolve_split(split_name):
    for field_name in SPLIT_FIELD_NAMES[split_name]:
        if field_name in graph_values:
            mask, selected_index, available_splits = select_mask(
                graph_values[field_name],
                SPLIT_INDEX,
                int(graph.num_nodes),
                field_name,
            )
            return {
                "source": field_name,
                "mask": mask,
                "selected_index": selected_index,
                "available_splits": available_splits,
            }

    split_dict = graph_values.get("split")
    if isinstance(split_dict, dict):
        for key in SPLIT_DICT_KEYS[split_name]:
            if key in split_dict:
                source_name = f"split.{key}"
                mask, selected_index, available_splits = select_mask(
                    split_dict[key],
                    SPLIT_INDEX,
                    int(graph.num_nodes),
                    source_name,
                )
                return {
                    "source": source_name,
                    "mask": mask,
                    "selected_index": selected_index,
                    "available_splits": available_splits,
                }
    return None


selected_splits = {
    split_name: split_info
    for split_name in ("train", "val", "test")
    if (split_info := resolve_split(split_name)) is not None
}

split_rows = []
for split_name, split_info in selected_splits.items():
    node_ids = torch.where(split_info["mask"])[0]
    sample_lengths = [len(str(node_texts[int(node_id)])) for node_id in node_ids]
    split_labels = (
        flat_labels[node_ids]
        if flat_labels is not None and node_ids.numel()
        else None
    )
    split_rows.append(
        {
            "划分": split_name,
            "来源字段": split_info["source"],
            "当前划分索引": split_info["selected_index"],
            "可用划分数": split_info["available_splits"],
            "节点数": int(node_ids.numel()),
            "标签类别数": (
                int(torch.unique(split_labels).numel())
                if split_labels is not None
                else None
            ),
            "最短文本字符数": min(sample_lengths) if sample_lengths else None,
            "平均文本字符数": (
                round(sum(sample_lengths) / len(sample_lengths), 2)
                if sample_lengths
                else None
            ),
            "最长文本字符数": max(sample_lengths) if sample_lengths else None,
        }
    )

if split_rows:
    split_overview = pd.DataFrame(split_rows)
    display(split_overview)
else:
    print("图对象中没有发现 train/val/test 划分字段。")

### train / val / test 标签分布

In [ ]:
split_label_rows = []
if flat_labels is not None:
    for split_name, split_info in selected_splits.items():
        node_ids = torch.where(split_info["mask"])[0]
        label_ids, label_counts = torch.unique(
            flat_labels[node_ids], sorted=True, return_counts=True
        )
        for label_id, count in zip(label_ids.tolist(), label_counts.tolist()):
            class_text = (
                str(class_texts[label_id])
                if 0 <= label_id < len(class_texts)
                else ""
            )
            split_label_rows.append(
                {
                    "划分": split_name,
                    "标签 ID": label_id,
                    "样本数": count,
                    "类别文本": class_text,
                }
            )

if split_label_rows:
    split_label_distribution = pd.DataFrame(split_label_rows)
    with pd.option_context("display.max_colwidth", 200):
        display(split_label_distribution)
else:
    print("没有可展示的划分标签。")

### train / val / test 样本预览

In [ ]:
split_sample_rows = []
for split_name, split_info in selected_splits.items():
    node_ids = torch.where(split_info["mask"])[0][:SAMPLE_SIZE]
    for node_id_tensor in node_ids:
        node_id = int(node_id_tensor)
        sample_text = str(node_texts[node_id])
        label_id = (
            int(flat_labels[node_id])
            if flat_labels is not None and node_id < flat_labels.numel()
            else None
        )
        class_text = (
            str(class_texts[label_id])
            if label_id is not None and 0 <= label_id < len(class_texts)
            else ""
        )
        split_sample_rows.append(
            {
                "划分": split_name,
                "节点 ID": node_id,
                "样本文本": sample_text,
                "文本字符数": len(sample_text),
                "空白分词数": len(sample_text.split()),
                "标签 ID": label_id,
                "类别文本": class_text,
            }
        )

split_samples = pd.DataFrame(split_sample_rows)
if split_samples.empty:
    print("没有可展示的划分样本。")
else:
    with pd.option_context("display.max_colwidth", 300):
        display(split_samples)

## 7. 边关系样例

In [ ]:
edge_index = getattr(graph, "edge_index", None)
if not isinstance(edge_index, torch.Tensor):
    print("图对象中没有 Tensor 类型的 edge_index。")
else:
    edge_index = edge_index.detach().cpu()
    edge_preview_size = min(SAMPLE_SIZE, edge_index.shape[1])
    edge_rows = [
        {"source": int(edge_index[0, index]), "target": int(edge_index[1, index])}
        for index in range(edge_preview_size)
    ]
    print(f"edge_index shape: {tuple(edge_index.shape)}")
    display(pd.DataFrame(edge_rows))

## 8. 任务映射

In [ ]:
def index_list(indices):
    if isinstance(indices, torch.Tensor):
        return indices.detach().cpu().reshape(-1).tolist()
    if hasattr(indices, "tolist"):
        indices = indices.tolist()
    if isinstance(indices, (list, tuple)):
        return list(indices)
    return [indices]


task_rows = []
for task_name, task_config in task_map.items():
    for role, (source_name, indices) in task_config.items():
        normalized_indices = index_list(indices)
        task_rows.append(
            {
                "任务": task_name,
                "角色": role,
                "来源特征组": source_name,
                "索引数量": len(normalized_indices),
                "索引预览": normalized_indices[:12],
            }
        )

display(pd.DataFrame(task_rows))

## 9. 加载器实现

In [ ]:
print(f"加载器文件: {dataset_module.__file__}\n")
print(inspect.getsource(dataset_module.get_data))

## 10. 一致性检查

In [ ]:
checks = [
    ("至少返回一个图", len(graphs) > 0),
    ("返回五组标准文本特征", len(text_features) == 5),
    ("节点文本数量等于节点数", len(node_texts) == int(graph.num_nodes)),
]

for task_name, task_config in task_map.items():
    for role, (source_name, indices) in task_config.items():
        source_values = text_feature_map.get(source_name)
        normalized_indices = index_list(indices)
        valid = source_values is not None and all(
            isinstance(index, int) and 0 <= index < len(source_values)
            for index in normalized_indices
        )
        checks.append((f"{task_name}.{role} 索引有效", valid))

check_table = pd.DataFrame(checks, columns=["检查项", "通过"])
display(check_table)

failed_checks = check_table.loc[~check_table["通过"], "检查项"].tolist()
if failed_checks:
    raise AssertionError(f"一致性检查失败: {failed_checks}")